# Text Feature Processing: Ad Creative Text Cleanup and Feature Engineering

This notebook processes ad creative text features by:
- Converting to lowercase
- Removing escape sequences (\n, \t, etc.)
- Removing URLs
- Normalizing whitespace
- Extracting and counting emojis
- Creating new derived features

## 1. Import Required Libraries

In [24]:
import pandas as pd
import numpy as np
import re
import emoji
import unicodedata
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore the Dataset

In [25]:
# Load the dataset
file_path = r"C:/Users/Vivobook/Documents/mm_detect/data/raw/ads_vietnam_clean.csv"
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head(3)

Dataset shape: (16678, 18)

Column names and types:
id                         object
page_id                    object
page_name                  object
ad_creation_time           object
ad_delivery_start_time     object
ad_delivery_stop_time      object
ad_creative_bodies         object
ad_creative_link_titles    object
currency                   object
impressions                object
spend                      object
target_gender              object
target_ages                object
target_locations           object
languages                  object
publisher_platforms        object
ad_snapshot_url            object
misinformation              int64
dtype: object

First few rows:


,id,page_id,page_name,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,ad_creative_bodies,ad_creative_link_titles,currency,impressions,spend,target_gender,target_ages,target_locations,languages,publisher_platforms,ad_snapshot_url,misinformation
0,823238210316661,102854402460581,FiberOne Boats,2025-11-10,2025-11-10,2025-11-25,['💓💓Bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi l...,['.'],NaN,NaN,NaN,Women,"['18', '65']","[{'name': 'Luân Đôn, Vương quốc Anh', 'num_obf...",['vi'],['facebook'],https://www.facebook.com/ads/archive/render_ad...,1
1,871132209123989,136028396253725,DramaBox-movies and drama 8,2026-02-08,2026-02-08,NaN,['Sau năm năm bị kẹt trong hôn nhân không tình...,['(Lồng tiếng)Cuộc đời hoàn hảo không cần anh'],NaN,NaN,NaN,All,"['18', '65']","[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network']",https://www.facebook.com/ads/archive/render_ad...,0
2,877151195021133,828292550372769,DramaBox-drama pendek,2026-02-06,2026-02-08,NaN,['Sau năm năm bị kẹt trong hôn nhân không tình...,['(Lồng tiếng)Cuộc đời hoàn hảo không cần anh'],NaN,NaN,NaN,All,"['18', '65']","[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network']",https://www.facebook.com/ads/archive/render_ad...,0


## 3. Clean Text Features: Convert to Lowercase and Remove Special Characters

In [26]:
def clean_text_basic(text):
    """
    Convert to lowercase and remove escape sequences like \n, \t, etc.
    """
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove escape sequences: \n, \r, \t, etc.
    text = text.replace('\\n', ' ')
    text = text.replace('\\r', ' ')
    text = text.replace('\\t', ' ')
    text = text.replace('\\xa0', ' ')
    
    # Remove other common escape sequences
    text = re.sub(r'\\[a-zA-Z]', ' ', text)
    
    return text

# Apply the cleaning function to both text features
print("Cleaning text features (lowercase, remove escape sequences)...")
df['ad_creative_bodies'] = df['ad_creative_bodies'].apply(clean_text_basic)
df['ad_creative_link_titles'] = df['ad_creative_link_titles'].apply(clean_text_basic)

print("Text cleaning completed!")
print("\nSample cleaned text:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles']].head())

Cleaning text features (lowercase, remove escape sequences)...
Text cleaning completed!

Sample cleaned text:
                                  ad_creative_bodies  \
0  ['💓💓bán lỗ  mẫu dép nữ tăng cao 6 cm cho mọi l...   
1  ['sau năm năm bị kẹt trong hôn nhân không tình...   
2  ['sau năm năm bị kẹt trong hôn nhân không tình...   
3  ['sau năm năm bị kẹt trong hôn nhân không tình...   
4  ['tái sinh, nam mạt trở nên lạnh lùng và quyết...   

                           ad_creative_link_titles  
0                                            ['.']  
1  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
2  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
3  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
4                     ['đích nữ tàn nhẫn đăng cơ']  


## 4. Remove URLs from Text Features

In [27]:
def remove_urls(text):
    """
    Remove URLs from text using regex pattern.
    Matches http, https, www, and other URL patterns.
    """
    if pd.isna(text):
        return ""
    
    # Remove URLs starting with http, https, www
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # Remove www. URLs
    text = re.sub(r'www\.(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    return text

# Apply URL removal
print("Removing URLs from text features...")
df['ad_creative_bodies'] = df['ad_creative_bodies'].apply(remove_urls)
df['ad_creative_link_titles'] = df['ad_creative_link_titles'].apply(remove_urls)

print("URL removal completed!")
print("\nSample text after URL removal:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles']].head())

Removing URLs from text features...
URL removal completed!

Sample text after URL removal:
                                  ad_creative_bodies  \
0  ['💓💓bán lỗ  mẫu dép nữ tăng cao 6 cm cho mọi l...   
1  ['sau năm năm bị kẹt trong hôn nhân không tình...   
2  ['sau năm năm bị kẹt trong hôn nhân không tình...   
3  ['sau năm năm bị kẹt trong hôn nhân không tình...   
4  ['tái sinh, nam mạt trở nên lạnh lùng và quyết...   

                           ad_creative_link_titles  
0                                            ['.']  
1  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
2  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
3  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
4                     ['đích nữ tàn nhẫn đăng cơ']  


## 5. Normalize Whitespace

In [28]:
def normalize_whitespace(text):
    """
    Normalize whitespace: remove extra spaces, tabs, and newlines.
    Replace multiple spaces with single space.
    """
    if pd.isna(text):
        return ""
    
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply whitespace normalization
print("Normalizing whitespace...")
df['ad_creative_bodies'] = df['ad_creative_bodies'].apply(normalize_whitespace)
df['ad_creative_link_titles'] = df['ad_creative_link_titles'].apply(normalize_whitespace)

print("Whitespace normalization completed!")
print("\nSample text after whitespace normalization:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles']].head())

Normalizing whitespace...
Whitespace normalization completed!

Sample text after whitespace normalization:
                                  ad_creative_bodies  \
0  ['💓💓bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứ...   
1  ['sau năm năm bị kẹt trong hôn nhân không tình...   
2  ['sau năm năm bị kẹt trong hôn nhân không tình...   
3  ['sau năm năm bị kẹt trong hôn nhân không tình...   
4  ['tái sinh, nam mạt trở nên lạnh lùng và quyết...   

                           ad_creative_link_titles  
0                                            ['.']  
1  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
2  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
3  ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']  
4                     ['đích nữ tàn nhẫn đăng cơ']  


## 6. Extract and Count Emojis

In [29]:
def extract_emojis(text):
    """
    Extract all emojis from text and return them as a concatenated string.
    Example: '💓💓👉👉👉'
    """
    if pd.isna(text):
        return ""
    
    # Extract emojis using emoji library
    emojis = ''.join(c for c in text if c in emoji.EMOJI_DATA)
    
    return emojis

def count_emojis(text):
    """
    Count the total number of emojis in the text.
    """
    if pd.isna(text):
        return 0
    
    # Count emojis using emoji library
    emoji_count = sum(1 for c in text if c in emoji.EMOJI_DATA)
    
    return emoji_count

# Extract emojis from both text features
print("Extracting emojis from text features...")

# Create temporary columns to store emojis from each feature
df['emoji_bodies'] = df['ad_creative_bodies'].apply(extract_emojis)
df['emoji_titles'] = df['ad_creative_link_titles'].apply(extract_emojis)

# Combine emojis from both features into one 'emoji' column
df['emoji'] = df['emoji_bodies'] + df['emoji_titles']

# Count total emojis
df['emoji_count'] = df['emoji'].apply(count_emojis)

print("Emoji extraction completed!")
print(f"\nTotal rows with emojis: {(df['emoji_count'] > 0).sum()}")
print("\nSample emojis extracted:")
print(df[df['emoji_count'] > 0][['ad_creative_bodies', 'ad_creative_link_titles', 'emoji', 'emoji_count']].head(10))

Extracting emojis from text features...
Emoji extraction completed!

Total rows with emojis: 8715

Sample emojis extracted:
                                   ad_creative_bodies  \
0   ['💓💓bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứ...   
13  ['dưới vòm trời bắc vực, những âm mưu đen tối ...   
22  ['làm cách nào để chồng tự nguyện rời xa người...   
26  ['⚡️trị #nám cấp tốc. 10 ngày hết nám là có th...   
27  ['💮 sạch nám, da mướt mịn, căng bóng ⏩giảm giá...   
36  ['💮 sạch nám, da mướt mịn, căng bóng ⏩giảm giá...   
37  ['dễ dàng thử thách mục tiêu của bạn 🚀 1️⃣làm ...   
41  ['⭐⭐ ưu đãi #mua_1_tặng_1 #mua_2_tặng_2⭐⭐ "𝗞𝗘𝗠...   
42  ['làm cách nào để chồng tự nguyện rời xa người...   
48  ['⚡️trị #nám chỉ với liệu trình ngắn => da xin...   

                              ad_creative_link_titles                 emoji  \
0                                               ['.']                 💓💓👉👉👉   
13                       ['luyện nhầm thần công👉️👉️']                    👉👉   
22         

## 7. Remove Emojis from Original Features

In [30]:
def remove_emojis(text):
    """
    Remove all emojis from text, keeping only text characters.
    """
    if pd.isna(text):
        return ""
    
    # Remove emojis using emoji library
    text = ''.join(c for c in text if c not in emoji.EMOJI_DATA)
    
    # Normalize whitespace again after emoji removal
    text = normalize_whitespace(text)
    
    return text

# Remove emojis from original text features
print("Removing emojis from original text features...")
df['ad_creative_bodies'] = df['ad_creative_bodies'].apply(remove_emojis)
df['ad_creative_link_titles'] = df['ad_creative_link_titles'].apply(remove_emojis)

# Clean up temporary emoji columns
df.drop(['emoji_bodies', 'emoji_titles'], axis=1, inplace=True)

print("Emoji removal completed!")
print("\nSample text after emoji removal:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles', 'emoji', 'emoji_count']].head(10))

Removing emojis from original text features...
Emoji removal completed!

Sample text after emoji removal:
                                  ad_creative_bodies  \
0  ['bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứa ...   
1  ['sau năm năm bị kẹt trong hôn nhân không tình...   
2  ['sau năm năm bị kẹt trong hôn nhân không tình...   
3  ['sau năm năm bị kẹt trong hôn nhân không tình...   
4  ['tái sinh, nam mạt trở nên lạnh lùng và quyết...   
5  ['sau năm năm bị kẹt trong hôn nhân không tình...   
6  ['tái sinh, nam mạt trở nên lạnh lùng và quyết...   
7  ['học bá lâm tư vọng xuất thân ở tỉnh sơn hà b...   
8  ['sau năm năm bị kẹt trong hôn nhân không tình...   
9  ['sau năm năm bị kẹt trong hôn nhân không tình...   

                             ad_creative_link_titles  emoji  emoji_count  
0                                              ['.']  💓💓👉👉👉            5  
1    ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']                   0  
2    ['(lồng tiếng)cuộc đời hoàn hảo không cần anh']

## 8. Calculate Text Length

In [31]:
# Calculate text length for both features
print("Calculating text length...")

df['ad_creative_bodies_length'] = df['ad_creative_bodies'].apply(lambda x: len(x) if pd.notna(x) else 0)
df['ad_creative_link_titles_length'] = df['ad_creative_link_titles'].apply(lambda x: len(x) if pd.notna(x) else 0)

# Create combined text_length feature (total length of both)
df['text_length'] = df['ad_creative_bodies_length'] + df['ad_creative_link_titles_length']

print("Text length calculation completed!")
print("\nText length statistics:")
print(df[['ad_creative_bodies_length', 'ad_creative_link_titles_length', 'text_length']].describe())
print("\nSample with text lengths:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles', 'ad_creative_bodies_length', 'ad_creative_link_titles_length', 'text_length']].head())

Calculating text length...
Text length calculation completed!

Text length statistics:
       ad_creative_bodies_length  ad_creative_link_titles_length   text_length
count               16678.000000                    16678.000000  16678.000000
mean                 2035.723948                       43.630231   2079.354179
std                  6904.961669                       66.717824   6906.198353
min                     0.000000                        0.000000      0.000000
25%                   208.000000                       22.000000    254.000000
50%                   330.000000                       35.000000    365.000000
75%                   645.000000                       50.000000    686.000000
max                 48785.000000                     1071.000000  48889.000000

Sample with text lengths:
                                  ad_creative_bodies  \
0  ['bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứa ...   
1  ['sau năm năm bị kẹt trong hôn nhân không tình...   
2  ['sa

## 9. Create New Features and Save Dataset

In [32]:
# Display summary of new features
print("=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)
print(f"\nOriginal dataset shape: {df.shape[0]} rows")
print(f"\nNew features created:")
print(f"  - 'emoji': Contains extracted emojis (e.g., '💓💓👉👉👉')")
print(f"  - 'emoji_count': Number of emojis in each row")
print(f"  - 'text_length': Total character length of both text features")
print(f"  - 'ad_creative_bodies_length': Character length of ad_creative_bodies")
print(f"  - 'ad_creative_link_titles_length': Character length of ad_creative_link_titles")

print(f"\nFinal dataset shape: {df.shape}")
print(f"\nNew columns in dataset:")
print(f"  {list(df.columns)}")

# Display summary statistics
print(f"\n" + "=" * 80)
print("FEATURE STATISTICS")
print("=" * 80)
print(f"\nRows with emojis: {(df['emoji_count'] > 0).sum()} ({(df['emoji_count'] > 0).sum() / len(df) * 100:.2f}%)")
print(f"Rows without emojis: {(df['emoji_count'] == 0).sum()} ({(df['emoji_count'] == 0).sum() / len(df) * 100:.2f}%)")
print(f"\nEmoji count statistics:")
print(df['emoji_count'].describe())
print(f"\nText length statistics:")
print(df['text_length'].describe())

# Save the processed dataset
output_path = r"C:/Users/Vivobook\Documents/mm_detect/ads_vietnam_processed.csv"
df.to_csv(output_path, index=False)
print(f"\n" + "=" * 80)
print(f"✓ Dataset saved to: {output_path}")
print("=" * 80)

PROCESSING SUMMARY

Original dataset shape: 16678 rows

New features created:
  - 'emoji': Contains extracted emojis (e.g., '💓💓👉👉👉')
  - 'emoji_count': Number of emojis in each row
  - 'text_length': Total character length of both text features
  - 'ad_creative_bodies_length': Character length of ad_creative_bodies
  - 'ad_creative_link_titles_length': Character length of ad_creative_link_titles

Final dataset shape: (16678, 23)

New columns in dataset:
  ['id', 'page_id', 'page_name', 'ad_creation_time', 'ad_delivery_start_time', 'ad_delivery_stop_time', 'ad_creative_bodies', 'ad_creative_link_titles', 'currency', 'impressions', 'spend', 'target_gender', 'target_ages', 'target_locations', 'languages', 'publisher_platforms', 'ad_snapshot_url', 'misinformation', 'emoji', 'emoji_count', 'ad_creative_bodies_length', 'ad_creative_link_titles_length', 'text_length']

FEATURE STATISTICS

Rows with emojis: 8715 (52.25%)
Rows without emojis: 7963 (47.75%)

Emoji count statistics:
count    1667

## 10. Final Data Preview

In [33]:
# Display final processed data with all relevant columns
print("Final processed data sample (with new features):")
cols_to_show = ['ad_creative_bodies', 'ad_creative_link_titles', 'emoji', 'emoji_count', 'text_length']
print(df[cols_to_show].head(15).to_string())

print("\n\nData types of key columns:")
print(df[['ad_creative_bodies', 'ad_creative_link_titles', 'emoji', 'emoji_count', 'text_length']].dtypes)

print("\n\nSample rows with emojis:")
emoji_sample = df[df['emoji_count'] > 0][cols_to_show].head(10)
if len(emoji_sample) > 0:
    print(emoji_sample.to_string())
else:
    print("No emojis found in the dataset")

Final processed data sample (with new features):
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          ad_creative_bodies                                 ad_creative_link_titles  emoji  emoji_count  text_length
0                                                                                                                                                                                     ['bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứa tuổi giảm tới 65% chỉ còn #179k, ship 20k ,có 2 màu đen và kem đặc biệt mua 2 đôi chỉ có #320k ,miễn ship -nhận hàng kiểm tra

In [34]:
psd = pd.read_csv("C:/Users/Vivobook\Documents/mm_detect/ads_vietnam_processed.csv")

psd.head(5)

,id,page_id,page_name,ad_creation_time,ad_delivery_start_time,ad_delivery_stop_time,ad_creative_bodies,ad_creative_link_titles,currency,impressions,...,target_locations,languages,publisher_platforms,ad_snapshot_url,misinformation,emoji,emoji_count,ad_creative_bodies_length,ad_creative_link_titles_length,text_length
0,823238210316661,102854402460581,FiberOne Boats,2025-11-10,2025-11-10,2025-11-25,['bán lỗ mẫu dép nữ tăng cao 6 cm cho mọi lứa ...,['.'],NaN,NaN,...,"[{'name': 'Luân Đôn, Vương quốc Anh', 'num_obf...",['vi'],['facebook'],https://www.facebook.com/ads/archive/render_ad...,1,💓💓👉👉👉,5,326,5,331
1,871132209123989,136028396253725,DramaBox-movies and drama 8,2026-02-08,2026-02-08,NaN,['sau năm năm bị kẹt trong hôn nhân không tình...,['(lồng tiếng)cuộc đời hoàn hảo không cần anh'],NaN,NaN,...,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network']",https://www.facebook.com/ads/archive/render_ad...,0,NaN,0,241,47,288
2,877151195021133,828292550372769,DramaBox-drama pendek,2026-02-06,2026-02-08,NaN,['sau năm năm bị kẹt trong hôn nhân không tình...,['(lồng tiếng)cuộc đời hoàn hảo không cần anh'],NaN,NaN,...,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network']",https://www.facebook.com/ads/archive/render_ad...,0,NaN,0,241,47,288
3,881581114655479,136028396253725,DramaBox-movies and drama 8,2026-02-08,2026-02-08,NaN,['sau năm năm bị kẹt trong hôn nhân không tình...,['(lồng tiếng)cuộc đời hoàn hảo không cần anh'],NaN,NaN,...,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network']",https://www.facebook.com/ads/archive/render_ad...,0,NaN,0,241,47,288
4,1227833952148512,580340045172078,Dramawave-New short dramas,2026-02-08,2026-02-08,2026-02-08,"['tái sinh, nam mạt trở nên lạnh lùng và quyết...",['đích nữ tàn nhẫn đăng cơ'],NaN,NaN,...,"[{'name': 'Worldwide', 'num_obfuscated': 0, 't...",['vi'],"['facebook', 'instagram', 'audience_network', ...",https://www.facebook.com/ads/archive/render_ad...,0,NaN,0,285,28,313


## 11. Time-Based Features: Ad Duration and Campaign Timing

In [35]:
# Reload the original dataset with all columns for feature engineering
original_df = pd.read_csv(r"C:\Users\Vivobook\Documents\mm_detect\ads_vietnam_clean.csv")

# Merge with previously processed text features and indicators
df_merged = original_df.copy()

# Time-based features
print("Creating time-based features...")

# Parse datetime columns
df_merged['ad_delivery_start_time'] = pd.to_datetime(df_merged['ad_delivery_start_time'], errors='coerce')
df_merged['ad_delivery_stop_time'] = pd.to_datetime(df_merged['ad_delivery_stop_time'], errors='coerce')

# Calculate ad duration in days
df_merged['ads_duration'] = (df_merged['ad_delivery_stop_time'] - df_merged['ad_delivery_start_time']).dt.days

# Calculate launch delay (days between current date and start time)
current_date = pd.Timestamp.now()
df_merged['launch_delay'] = (df_merged['ad_delivery_start_time'] - current_date).dt.days

# Calculate burstiness (1 = burst campaign if duration < 7 days, 0 = sustained)
df_merged['burstiness'] = (df_merged['ads_duration'] < 7).astype(int)

# Active status (1 = active, 0 = inactive based on stop time)
df_merged['active_status'] = (df_merged['ad_delivery_stop_time'] > current_date).astype(int)

print("Time-based features created!")
print(f"\nTime feature statistics:")
print(df_merged[['ads_duration', 'launch_delay', 'burstiness', 'active_status']].describe())

Creating time-based features...
Time-based features created!

Time feature statistics:
       ads_duration  launch_delay    burstiness  active_status
count  13633.000000  16678.000000  16678.000000        16678.0
mean       2.095577    -88.429908      0.765859            0.0
std        5.087530     51.612684      0.423473            0.0
min        0.000000   -471.000000      0.000000            0.0
25%        0.000000   -100.000000      1.000000            0.0
50%        1.000000    -70.000000      1.000000            0.0
75%        2.000000    -63.000000      1.000000            0.0
max      256.000000    -59.000000      1.000000            0.0


## 12. Page-Based Features: Campaign Behavior and Page Characteristics

In [36]:
print("Creating page-based features...")

# Ads per page - campaign behavior
ads_per_page = df_merged.groupby('page_id').size().reset_index(name='ads_per_page')
df_merged = df_merged.merge(ads_per_page, on='page_id', how='left')

# Average ad duration per page - stability
avg_ad_duration = df_merged.groupby('page_id')['ads_duration'].mean().reset_index(name='avg_ad_duration')
df_merged = df_merged.merge(avg_ad_duration, on='page_id', how='left')

# Repeated text ratio - template scams
def calculate_repeated_text_ratio(page_id):
    page_texts = df_merged[df_merged['page_id'] == page_id]['ad_creative_bodies'].values
    if len(page_texts) <= 1:
        return 0
    from collections import Counter
    text_counts = Counter(page_texts)
    most_common_count = text_counts.most_common(1)[0][1] if text_counts else 0
    ratio = most_common_count / len(page_texts)
    return ratio

repeated_text_ratio = df_merged.groupby('page_id')['page_id'].apply(
    lambda x: calculate_repeated_text_ratio(x.iloc[0])
).reset_index(name='repeated_text_ratio')
df_merged = df_merged.merge(repeated_text_ratio, on='page_id', how='left')

# Historical volume - fly-by-night pages (count of ads per page)
historical_volume = df_merged.groupby('page_id').size().reset_index(name='historical_volume')
df_merged = df_merged.merge(historical_volume, on='page_id', how='left')

print("Page-based features created!")
print(f"\nPage feature statistics:")
print(df_merged[['ads_per_page', 'avg_ad_duration', 'repeated_text_ratio', 'historical_volume']].describe())

Creating page-based features...
Page-based features created!

Page feature statistics:
       ads_per_page  avg_ad_duration  repeated_text_ratio  historical_volume
count  16678.000000     15706.000000         16678.000000       16678.000000
mean      65.157213         2.040427             0.600372          65.157213
std      100.633602         4.085667             0.330290         100.633602
min        1.000000         0.000000             0.000000           1.000000
25%        6.000000         0.633333             0.333333           6.000000
50%       25.000000         1.000000             0.605263          25.000000
75%       80.000000         1.933333             0.952381          80.000000
max      467.000000        74.750000             1.000000         467.000000


## 13. Spend-Based Features: Campaign Financial Metrics

In [37]:
print("Creating spend-based features...")

# Convert spend to numeric, handling NaN values
df_merged['spend'] = pd.to_numeric(df_merged['spend'], errors='coerce')
df_merged['impressions'] = pd.to_numeric(df_merged['impressions'], errors='coerce')

# Spend per day - aggressive push
df_merged['spend_per_day'] = df_merged['spend'] / (df_merged['ads_duration'].clip(lower=1))

# Impressions per day - viral push
df_merged['impressions_per_day'] = df_merged['impressions'] / (df_merged['ads_duration'].clip(lower=1))

# CPM estimate (Cost Per Mille) - anomalies
# CPM = (spend / impressions) * 1000
df_merged['CPM_estimate'] = (df_merged['spend'] / df_merged['impressions'].clip(lower=1)) * 1000

# Low spend high reach - suspicious (high impressions with low spend)
# Define as: impressions_per_day > median and spend_per_day < median
median_impressions_per_day = df_merged['impressions_per_day'].median()
median_spend_per_day = df_merged['spend_per_day'].median()

df_merged['low_spend_high_reach'] = (
    (df_merged['impressions_per_day'] > median_impressions_per_day) & 
    (df_merged['spend_per_day'] < median_spend_per_day)
).astype(int)

print("Spend-based features created!")
print(f"\nSpend feature statistics:")
print(df_merged[['spend_per_day', 'impressions_per_day', 'CPM_estimate', 'low_spend_high_reach']].describe())

Creating spend-based features...
Spend-based features created!

Spend feature statistics:
       spend_per_day  impressions_per_day  CPM_estimate  low_spend_high_reach
count            0.0                  0.0           0.0               16678.0
mean             NaN                  NaN           NaN                   0.0
std              NaN                  NaN           NaN                   0.0
min              NaN                  NaN           NaN                   0.0
25%              NaN                  NaN           NaN                   0.0
50%              NaN                  NaN           NaN                   0.0
75%              NaN                  NaN           NaN                   0.0
max              NaN                  NaN           NaN                   0.0


## 14. Demographic Features: Targeting Characteristics

In [38]:
print("Creating demographic features...")

# Age span - generic scams
# Parse age ranges like "18-24" and calculate span
def extract_age_span(age_str):
    if pd.isna(age_str):
        return 0
    try:
        ages = str(age_str).split('-')
        if len(ages) == 2:
            return int(ages[1]) - int(ages[0])
    except:
        return 0
    return 0

df_merged['age_span'] = df_merged['target_ages'].apply(extract_age_span)

# Number of countries - cross-border
def count_countries(location_str):
    if pd.isna(location_str):
        return 0
    return len(str(location_str).split(',')) if str(location_str).strip() else 0

df_merged['num_countries'] = df_merged['target_locations'].apply(count_countries)

# Gender specific targeting
def create_gender_flags(gender_str):
    if pd.isna(gender_str):
        return 0, 0, 0
    gender_str = str(gender_str).lower()
    women_targeted = 1 if 'women' in gender_str or 'female' in gender_str else 0
    men_targeted = 1 if 'men' in gender_str or 'male' in gender_str else 0
    all_targeted = 1 if 'all' in gender_str or (women_targeted == 0 and men_targeted == 0) else 0
    return women_targeted, men_targeted, all_targeted

df_merged[['women_targeted', 'men_targeted', 'all_targeted']] = df_merged['target_gender'].apply(
    lambda x: pd.Series(create_gender_flags(x))
)

# Language-location mismatch - red flag
# Simple check: if languages present but locations not, or vice versa
def check_language_location_mismatch(row):
    langs_present = 1 if pd.notna(row['languages']) and str(row['languages']).strip() else 0
    locs_present = 1 if pd.notna(row['target_locations']) and str(row['target_locations']).strip() else 0
    # Mismatch if only one is present
    return 1 if (langs_present + locs_present == 1) else 0

df_merged['language_location_mismatch'] = df_merged.apply(check_language_location_mismatch, axis=1)

print("Demographic features created!")
print(f"\nDemographic feature statistics:")
print(df_merged[['age_span', 'num_countries', 'women_targeted', 'men_targeted', 'all_targeted', 'language_location_mismatch']].describe())

Creating demographic features...
Demographic features created!

Demographic feature statistics:
       age_span  num_countries  women_targeted  men_targeted  all_targeted  \
count   16678.0   16678.000000    16678.000000  16678.000000  16678.000000   
mean        0.0      31.023204        0.207279      0.294220      0.675501   
std         0.0      44.859968        0.405369      0.455705      0.468202   
min         0.0       0.000000        0.000000      0.000000      0.000000   
25%         0.0       4.000000        0.000000      0.000000      0.000000   
50%         0.0       4.000000        0.000000      0.000000      1.000000   
75%         0.0      60.000000        0.000000      1.000000      1.000000   
max         0.0     400.000000        1.000000      1.000000      1.000000   

       language_location_mismatch  
count                16678.000000  
mean                     0.105648  
std                      0.307396  
min                      0.000000  
25%                  

## 15. Platform Features: Publisher Platform Characteristics

In [39]:
print("Creating platform features...")

# Platform count - broad vs niche
def count_platforms(platform_str):
    if pd.isna(platform_str):
        return 0
    return len(str(platform_str).split(','))

df_merged['platform_count'] = df_merged['publisher_platforms'].apply(count_platforms)

# FB only flag - older audience targeting
def check_fb_only(platform_str):
    if pd.isna(platform_str):
        return 0
    platforms = str(platform_str).lower()
    return 1 if 'facebook' in platforms and 'instagram' not in platforms else 0

df_merged['FB_only_flag'] = df_merged['publisher_platforms'].apply(check_fb_only)

# IG only flag - visual deception
def check_ig_only(platform_str):
    if pd.isna(platform_str):
        return 0
    platforms = str(platform_str).lower()
    return 1 if 'instagram' in platforms and 'facebook' not in platforms else 0

df_merged['IG_only_flag'] = df_merged['publisher_platforms'].apply(check_ig_only)

print("Platform features created!")
print(f"\nPlatform feature statistics:")
print(df_merged[['platform_count', 'FB_only_flag', 'IG_only_flag']].describe())

# Display feature engineering summary
print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"\nTotal rows: {len(df_merged)}")
print(f"Total new features created: {len([c for c in df_merged.columns if c not in original_df.columns]) - 1}")
print(f"\nNew engineered columns:")
engineered_cols = [c for c in df_merged.columns if c not in original_df.columns]
for i, col in enumerate(engineered_cols, 1):
    print(f"  {i:2d}. {col}")

Creating platform features...
Platform features created!

Platform feature statistics:
       platform_count  FB_only_flag  IG_only_flag
count    16678.000000  16678.000000  16678.000000
mean         3.452212      0.256805      0.004077
std          1.570540      0.436884      0.063725
min          1.000000      0.000000      0.000000
25%          2.000000      0.000000      0.000000
50%          4.000000      0.000000      0.000000
75%          5.000000      1.000000      0.000000
max          5.000000      1.000000      1.000000

FEATURE ENGINEERING SUMMARY

Total rows: 16678
Total new features created: 20

New engineered columns:
   1. ads_duration
   2. launch_delay
   3. burstiness
   4. active_status
   5. ads_per_page
   6. avg_ad_duration
   7. repeated_text_ratio
   8. historical_volume
   9. spend_per_day
  10. impressions_per_day
  11. CPM_estimate
  12. low_spend_high_reach
  13. age_span
  14. num_countries
  15. women_targeted
  16. men_targeted
  17. all_targeted
  18. l

## 16. Save Engineered Dataset and Prepare for Correlation Analysis

In [40]:
# Save the complete engineered dataset
engineered_output_path = r"C:\Users\Vivobook\Documents\mm_detect\ads_vietnam_engineered_features.csv"
df_merged.to_csv(engineered_output_path, index=False)

print(f"✓ Engineered dataset saved to: {engineered_output_path}")
print(f"\nDataset shape: {df_merged.shape}")
print(f"Columns: {df_merged.shape[1]}")

# Select numeric columns for correlation analysis
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns available for correlation: {len(numeric_cols)}")
print(f"Numeric columns: {numeric_cols}")

✓ Engineered dataset saved to: C:\Users\Vivobook\Documents\mm_detect\ads_vietnam_engineered_features.csv

Dataset shape: (16678, 39)
Columns: 39

Numeric columns available for correlation: 24
Numeric columns: ['impressions', 'spend', 'misinformation', 'ads_duration', 'launch_delay', 'burstiness', 'active_status', 'ads_per_page', 'avg_ad_duration', 'repeated_text_ratio', 'historical_volume', 'spend_per_day', 'impressions_per_day', 'CPM_estimate', 'low_spend_high_reach', 'age_span', 'num_countries', 'women_targeted', 'men_targeted', 'all_targeted', 'language_location_mismatch', 'platform_count', 'FB_only_flag', 'IG_only_flag']


## 17. Correlation Analysis and Visualizations

In [41]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

print("Computing correlation matrix...")

# Select engineered features for correlation analysis
engineered_features = [
    'ads_duration', 'launch_delay', 'burstiness', 'active_status',
    'ads_per_page', 'avg_ad_duration', 'repeated_text_ratio', 'historical_volume',
    'spend_per_day', 'impressions_per_day', 'CPM_estimate', 'low_spend_high_reach',
    'age_span', 'num_countries', 'women_targeted', 'men_targeted', 'all_targeted',
    'language_location_mismatch', 'platform_count', 'FB_only_flag', 'IG_only_flag',
    'text_length', 'emoji_count'
]

# Filter to only available features
available_features = [col for col in engineered_features if col in df_merged.columns]

# Create correlation matrix
corr_data = df_merged[available_features].fillna(0)
correlation_matrix = corr_data.corr(method='pearson')

print(f"Correlation matrix computed ({len(available_features)} features)")

# 1. Full Correlation Heatmap
print("\nGenerating full correlation heatmap...")
plt.figure(figsize=(20, 16))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - All Engineered Features', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
heatmap_path = r"C:\Users\Vivobook\Documents\mm_detect\outputs\figures\correlation_heatmap.png"
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Heatmap saved: {heatmap_path}")

# 2. Category-wise Correlations
print("\nGenerating category-wise correlations...")
categories = {
    'Time-based': ['ads_duration', 'launch_delay', 'burstiness', 'active_status'],
    'Page-based': ['ads_per_page', 'avg_ad_duration', 'repeated_text_ratio', 'historical_volume'],
    'Spend-based': ['spend_per_day', 'impressions_per_day', 'CPM_estimate', 'low_spend_high_reach'],
    'Demographic': ['age_span', 'num_countries', 'women_targeted', 'men_targeted', 'all_targeted', 'language_location_mismatch'],
    'Platform': ['platform_count', 'FB_only_flag', 'IG_only_flag'],
    'Text': ['text_length', 'emoji_count']
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Category-wise Feature Correlations', fontsize=16, fontweight='bold')

for idx, (category, cols) in enumerate(categories.items()):
    ax = axes[idx // 3, idx % 3]
    available_cat_cols = [col for col in cols if col in available_features]
    
    if len(available_cat_cols) > 0:
        cat_corr = df_merged[available_cat_cols].corr()
        sns.heatmap(cat_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                   ax=ax, square=True, cbar_kws={"shrink": 0.8})
        ax.set_title(f'{category} Features', fontweight='bold')

plt.tight_layout()
category_corr_path = r"C:\Users\Vivobook\Documents\mm_detect\outputs\figures\category_correlations.png"
plt.savefig(category_corr_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Category correlations saved: {category_corr_path}")

print("\nCorrelation analysis completed!")

Computing correlation matrix...
Correlation matrix computed (21 features)

Generating full correlation heatmap...
✓ Heatmap saved: C:\Users\Vivobook\Documents\mm_detect\outputs\figures\correlation_heatmap.png

Generating category-wise correlations...
✓ Category correlations saved: C:\Users\Vivobook\Documents\mm_detect\outputs\figures\category_correlations.png

Correlation analysis completed!


In [42]:
# 3. Top Correlations Analysis
print("Analyzing top correlations...")

# Extract upper triangle of correlation matrix
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_pairs.append({
            'Feature 1': correlation_matrix.columns[i],
            'Feature 2': correlation_matrix.columns[j],
            'Correlation': correlation_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs)
corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
top_correlations = corr_df.nlargest(20, 'Abs_Correlation')

print("\nTop 20 Strongest Correlations:")
print("="*80)
for idx, row in top_correlations.iterrows():
    print(f"{row['Feature 1']:25} <-> {row['Feature 2']:25} | {row['Correlation']:+.4f}")

# Visualize top correlations
fig, ax = plt.subplots(figsize=(14, 10))
y_pos = np.arange(len(top_correlations))
colors = ['red' if x < 0 else 'green' for x in top_correlations['Correlation']]

ax.barh(y_pos, top_correlations['Correlation'], color=colors, alpha=0.7)

# Create labels
labels = [f"{row['Feature 1'][:15]}...\nvs\n{row['Feature 2'][:15]}..." 
          if len(row['Feature 1']) > 15 or len(row['Feature 2']) > 15 
          else f"{row['Feature 1']}\nvs\n{row['Feature 2']}"
          for _, row in top_correlations.iterrows()]

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Correlation Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Top 20 Strongest Feature Correlations', fontsize=13, fontweight='bold', pad=20)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

# Add correlation values on bars
for i, v in enumerate(top_correlations['Correlation']):
    ax.text(v + 0.01 if v > 0 else v - 0.01, i, f'{v:.3f}', 
            va='center', ha='left' if v > 0 else 'right', fontsize=9, fontweight='bold')

plt.tight_layout()
top_corr_path = r"C:\Users\Vivobook\Documents\mm_detect\outputs\figures\top_correlations.png"
plt.savefig(top_corr_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n✓ Top correlations visualization saved: {top_corr_path}")

# Create detailed correlation report
print("\n" + "="*80)
print("CORRELATION STATISTICS")
print("="*80)

# Positive and negative correlations
positive_corr = corr_df[corr_df['Correlation'] > 0]
negative_corr = corr_df[corr_df['Correlation'] < 0]

print(f"\nTotal correlation pairs: {len(corr_df)}")
print(f"Positive correlations: {len(positive_corr)} ({len(positive_corr)/len(corr_df)*100:.1f}%)")
print(f"Negative correlations: {len(negative_corr)} ({len(negative_corr)/len(corr_df)*100:.1f}%)")

print(f"\nCorrelation strength distribution:")
very_strong = len(corr_df[corr_df['Abs_Correlation'] > 0.7])
strong = len(corr_df[(corr_df['Abs_Correlation'] > 0.5) & (corr_df['Abs_Correlation'] <= 0.7)])
moderate = len(corr_df[(corr_df['Abs_Correlation'] > 0.3) & (corr_df['Abs_Correlation'] <= 0.5)])
weak = len(corr_df[corr_df['Abs_Correlation'] <= 0.3])

print(f"  Very strong (>0.7):  {very_strong:4d} pairs ({very_strong/len(corr_df)*100:5.1f}%)")
print(f"  Strong (0.5-0.7):    {strong:4d} pairs ({strong/len(corr_df)*100:5.1f}%)")
print(f"  Moderate (0.3-0.5):  {moderate:4d} pairs ({moderate/len(corr_df)*100:5.1f}%)")
print(f"  Weak (<0.3):         {weak:4d} pairs ({weak/len(corr_df)*100:5.1f}%)")

Analyzing top correlations...

Top 20 Strongest Correlations:
ads_per_page              <-> historical_volume         | +1.0000
men_targeted              <-> all_targeted              | -0.9316
women_targeted            <-> men_targeted              | +0.7920
platform_count            <-> FB_only_flag              | -0.7713
ads_duration              <-> avg_ad_duration           | +0.7669
women_targeted            <-> all_targeted              | -0.7378
num_countries             <-> platform_count            | +0.3577
num_countries             <-> FB_only_flag              | -0.3469
launch_delay              <-> avg_ad_duration           | -0.3135
ads_duration              <-> launch_delay              | -0.2961
all_targeted              <-> platform_count            | +0.2819
all_targeted              <-> FB_only_flag              | -0.2522
num_countries             <-> all_targeted              | +0.2440
burstiness                <-> avg_ad_duration           | -0.2197
num_countries 

## 18. Summary and Output Files

In [43]:
print("\n" + "="*80)
print("COMPLETE FEATURE ENGINEERING PIPELINE - FINAL SUMMARY")
print("="*80)

print("\n✓ FEATURE ENGINEERING COMPLETED\n")

print("New Feature Categories Created:")
print("-" * 80)

print("\n1. TIME-BASED FEATURES (Timing & Campaign Duration):")
time_features = ['ads_duration', 'launch_delay', 'burstiness', 'active_status']
for i, feat in enumerate(time_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Campaign timing indicator")

print("\n2. PAGE-BASED FEATURES (Campaign Behavior & Page Characteristics):")
page_features = ['ads_per_page', 'avg_ad_duration', 'repeated_text_ratio', 'historical_volume']
for i, feat in enumerate(page_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Page activity pattern")

print("\n3. SPEND-BASED FEATURES (Financial Metrics):")
spend_features = ['spend_per_day', 'impressions_per_day', 'CPM_estimate', 'low_spend_high_reach']
for i, feat in enumerate(spend_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Financial/reach metric")

print("\n4. DEMOGRAPHIC FEATURES (Targeting Characteristics):")
demo_features = ['age_span', 'num_countries', 'women_targeted', 'men_targeted', 'all_targeted', 'language_location_mismatch']
for i, feat in enumerate(demo_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Demographic targeting")

print("\n5. PLATFORM FEATURES (Publisher Platform Characteristics):")
platform_features = ['platform_count', 'FB_only_flag', 'IG_only_flag']
for i, feat in enumerate(platform_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Platform strategy")

print("\n6. TEXT FEATURES (Text Metrics):")
text_features = ['text_length', 'emoji_count']
for i, feat in enumerate(text_features, 1):
    if feat in df_merged.columns:
        print(f"   {i}. {feat:30} - Text characteristics")

print("\n" + "="*80)
print("OUTPUT FILES GENERATED:")
print("="*80)

output_files = [
    ("ads_vietnam_engineered_features.csv", "Complete dataset with all engineered features"),
    ("correlation_heatmap.png", "Full correlation matrix visualization"),
    ("category_correlations.png", "Category-wise feature correlations"),
    ("top_correlations.png", "Top 20 strongest feature correlations")
]

for i, (filename, description) in enumerate(output_files, 1):
    filepath = rf"C:\Users\Vivobook\Documents\mm_detect\notebooks\{filename}"
    print(f"\n{i}. {filename}")
    print(f"   Location: {filepath}")
    print(f"   Purpose: {description}")

print("\n" + "="*80)
print("FEATURE STATISTICS:")
print("="*80)
print(f"\nOriginal dataset shape: {original_df.shape}")
print(f"Engineered dataset shape: {df_merged.shape}")
print(f"New features engineered: {df_merged.shape[1] - original_df.shape[1]}")
print(f"Total numeric features for analysis: {len(numeric_cols)}")

print("\n" + "="*80)
print("✓ ALL PROCESSING COMPLETE!")
print("="*80)


COMPLETE FEATURE ENGINEERING PIPELINE - FINAL SUMMARY

✓ FEATURE ENGINEERING COMPLETED

New Feature Categories Created:
--------------------------------------------------------------------------------

1. TIME-BASED FEATURES (Timing & Campaign Duration):
   1. ads_duration                   - Campaign timing indicator
   2. launch_delay                   - Campaign timing indicator
   3. burstiness                     - Campaign timing indicator
   4. active_status                  - Campaign timing indicator

2. PAGE-BASED FEATURES (Campaign Behavior & Page Characteristics):
   1. ads_per_page                   - Page activity pattern
   2. avg_ad_duration                - Page activity pattern
   3. repeated_text_ratio            - Page activity pattern
   4. historical_volume              - Page activity pattern

3. SPEND-BASED FEATURES (Financial Metrics):
   1. spend_per_day                  - Financial/reach metric
   2. impressions_per_day            - Financial/reach metric
  

# 10. Save Train/Validation/Test Splits

Split the preprocessed data into train, validation, and test sets with stratification based on the target label. All splits are saved as separate CSV files.

In [46]:
# Import necessary modules
from sklearn.model_selection import train_test_split
from pathlib import Path
import json
import os

# Use absolute path for reliability
base_dir = Path(r"C:\Users\Vivobook\Documents\mm_detect")
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"Output directory: {processed_dir}")
print(f"Output directory exists: {processed_dir.exists()}")

print("\nCreating stratified train/validation/test splits...")
print(f"Original dataset size: {len(df)}")

# Define split ratios (configurable from config if available)
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

# Check if we have a label column for stratification
stratify_col = None
if 'misinformation' in df.columns:
    stratify_col = df['misinformation']
    print(f"Using 'misinformation' column for stratified split")
elif 'label' in df.columns:
    stratify_col = df['label']
    print(f"Using 'label' column for stratified split")
else:
    print(f"No label column found, performing random split")

# First split: train+val vs test (85% vs 15%)
train_val_df, test_df = train_test_split(
    df,
    test_size=test_ratio,
    random_state=42,
    stratify=stratify_col
)

# Second split: train vs val from train_val (70/85 ≈ 82.4% vs 17.6%)
val_test_ratio = val_ratio / (train_ratio + val_ratio)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_test_ratio,
    random_state=42,
    stratify=train_val_df[stratify_col.name] if stratify_col is not None else None
)

# Display split statistics
print("\n" + "="*80)
print("TRAIN/VALIDATION/TEST SPLIT SUMMARY")
print("="*80)
print(f"Train set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validation set: {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test set: {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)} samples")

# Display class distribution per split
if stratify_col is not None:
    label_col = stratify_col.name
    print(f"\nClass distribution per split ({label_col}):")
    print(f"  Train: {dict(train_df[label_col].value_counts())}")
    print(f"  Validation: {dict(val_df[label_col].value_counts())}")
    print(f"  Test: {dict(test_df[label_col].value_counts())}")

# Save splits as CSV files using absolute paths
train_file = processed_dir / 'train.csv'
val_file = processed_dir / 'val.csv'
test_file = processed_dir / 'test.csv'

train_df.to_csv(str(train_file), index=False)
val_df.to_csv(str(val_file), index=False)
test_df.to_csv(str(test_file), index=False)

print(f"\n✓ Saved train.csv: {train_file}")
print(f"✓ Saved val.csv: {val_file}")
print(f"✓ Saved test.csv: {test_file}")

# Verify files were created
print("\n" + "="*80)
print("FILE VERIFICATION")
print("="*80)
print(f"train.csv exists: {train_file.exists()} (size: {train_file.stat().st_size if train_file.exists() else 0} bytes)")
print(f"val.csv exists: {val_file.exists()} (size: {val_file.stat().st_size if val_file.exists() else 0} bytes)")
print(f"test.csv exists: {test_file.exists()} (size: {test_file.stat().st_size if test_file.exists() else 0} bytes)")

# Save split metadata
split_metadata = {
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'total_size': len(df),
    'train_ratio': len(train_df) / len(df),
    'val_ratio': len(val_df) / len(df),
    'test_ratio': len(test_df) / len(df),
    'random_seed': 42,
    'stratify_column': stratify_col.name if stratify_col is not None else None,
    'columns': df.columns.tolist(),
    'output_directory': str(processed_dir)
}

metadata_file = processed_dir / 'split_metadata.json'
with open(str(metadata_file), 'w') as f:
    json.dump(split_metadata, f, indent=2)

print(f"✓ Saved split metadata: {metadata_file}")
print(f"✓ split_metadata.json exists: {metadata_file.exists()}")
print(f"\n✓ Preprocessing and splitting complete!")

Working directory: c:\Users\Vivobook\Documents\mm_detect\notebooks
Output directory: C:\Users\Vivobook\Documents\mm_detect\data\processed
Output directory exists: True

Creating stratified train/validation/test splits...
Original dataset size: 16678
Using 'misinformation' column for stratified split

TRAIN/VALIDATION/TEST SPLIT SUMMARY
Train set: 11674 samples (70.0%)
Validation set: 2502 samples (15.0%)
Test set: 2502 samples (15.0%)
Total: 16678 samples

Class distribution per split (misinformation):
  Train: {1: np.int64(8918), 0: np.int64(2756)}
  Validation: {1: np.int64(1911), 0: np.int64(591)}
  Test: {1: np.int64(1911), 0: np.int64(591)}

✓ Saved train.csv: C:\Users\Vivobook\Documents\mm_detect\data\processed\train.csv
✓ Saved val.csv: C:\Users\Vivobook\Documents\mm_detect\data\processed\val.csv
✓ Saved test.csv: C:\Users\Vivobook\Documents\mm_detect\data\processed\test.csv

FILE VERIFICATION
train.csv exists: True (size: 40104155 bytes)
val.csv exists: True (size: 8583849 byte